# 🚦 Notebook 0: Why Rate Limiting? (The Big Picture)

Before we learn algorithms, let's see **why** we need them.

Imagine your login API. A friendly user hits it ~1 time per login.
An attacker hits it **10,000 times per second** trying passwords.
Same endpoint, wildly different traffic — and if your server tries to handle
all of it, legitimate users suffer (slow responses, timeouts, downtime).

**Rate limiting** = *how many requests per unit of time* we allow from a
single client (IP, user, API key…). Beyond that, we **reject** or **delay**.


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 📖 Vocabulary you'll hear in the real world

| Term | Meaning | Example |
|---|---|---|
| **Rate limiting** | Short-window cap: "max 100 req / minute" | Login endpoint |
| **Throttling** | Slowing a client down (queue / delay) instead of rejecting | Background jobs |
| **Quota** | Long-window cap: "max 1M req / month" | Paid API tiers |
| **Backpressure** | Telling upstream "slow down!" via signals (429, queue depth) | Kafka consumers |

They overlap a lot. The common goal: **protect the service and keep it fair.**


## 🌍 Real-world limits (approximate, at time of writing)

You'll rarely pick limits from thin air — borrow from the giants:

| Service | Policy | Algorithm (public info) |
|---|---|---|
| **GitHub REST API** | 60 req/hr unauthenticated, 5,000 req/hr authenticated | Fixed-window counter + `X-RateLimit-*` headers |
| **GitHub GraphQL** | 5,000 **points**/hr (query cost varies) | Cost-based token bucket |
| **Stripe API** | 100 read + 100 write req/sec, burstable | Token bucket |
| **Discord Bot API** | 50 req/sec global + per-route buckets | Token bucket per bucket |
| **Twitter/X API v2** | Per-endpoint: e.g. 300 req / 15 min | Fixed window |
| **AWS API Gateway** | Default 10,000 req/sec account-wide, 5,000 burst | Token bucket |
| **Cloudflare** | Customer-defined; internal edge uses a hybrid sliding window | Sliding-window counter (O(1)) |

These are **examples**, not contracts — vendors change them. But notice the
pattern: everyone ends up with **token bucket for user-facing quotas** and
**sliding-window / leaky bucket for traffic shaping**. We'll build both.


## ❌ Bad: no limiter at all

Let's simulate a single shared resource (our "server") handling a burst of
requests. Without a limiter, *every* request goes through — including the
abusive ones. Legit users share the pain.


In [ ]:
import time

def handle_request(req_id):
    # Pretend each request costs 10ms of CPU on the server.
    time.sleep(0.01)
    return f"ok {req_id}"

# 1 legit user sends 5 requests; 1 abusive script sends 200.
legit = [("legit", i) for i in range(5)]
abuser = [("abuser", i) for i in range(200)]
traffic = legit + abuser  # pretend they arrive roughly together

start = time.monotonic()
served = 0
for who, i in traffic:
    handle_request(i)
    served += 1
elapsed = time.monotonic() - start
print(f"served {served} requests in {elapsed:.2f}s")
print("-> legit users waited behind the abuser. Everyone is slow.")

## ✅ Good: add a limiter *per client*

The fix is to track usage **per client key** (IP, user id, API key) and
reject once a client exceeds its budget. Below is the simplest possible
limiter: a **fixed counter per key per second**. We'll see its flaws in
later notebooks and build better ones — this is just to feel the shape of
the solution.


In [ ]:
import time
from collections import defaultdict

class NaivePerKeyLimiter:
    def __init__(self, limit_per_sec):
        self.limit = limit_per_sec
        self.counts = defaultdict(int)
        self.window_start = time.monotonic()

    def allow(self, key):
        now = time.monotonic()
        if now - self.window_start >= 1.0:
            self.counts.clear()
            self.window_start = now
        if self.counts[key] < self.limit:
            self.counts[key] += 1
            return True
        return False

limiter = NaivePerKeyLimiter(limit_per_sec=10)

start = time.monotonic()
legit_ok = abuser_ok = 0
for who, i in traffic:
    if limiter.allow(who):
        handle_request(i)
        if who == "legit":
            legit_ok += 1
        else:
            abuser_ok += 1

elapsed = time.monotonic() - start
print(f"elapsed: {elapsed:.2f}s")
print(f"legit served: {legit_ok}/5")
print(f"abuser served: {abuser_ok}/200 (capped at ~10/s)")
print("-> legit user is no longer blocked by the abuser.")

## 🌐 How the internet does it: HTTP `429 Too Many Requests`

When you reject a request, tell the client **why** and **when to retry**.
The standard response is:

- HTTP status **429 Too Many Requests**
- Header **`Retry-After: <seconds>`** (or an HTTP date)
- Optional headers many APIs send (GitHub, Stripe, Discord):
  - `X-RateLimit-Limit`    — your ceiling (e.g., `60`)
  - `X-RateLimit-Remaining` — tokens left (e.g., `12`)
  - `X-RateLimit-Reset`     — unix time when it resets

A well-behaved client reads `Retry-After` and **waits that long** before
retrying — see notebook 4 for exponential backoff + jitter.


In [ ]:
# Tiny demo: format a 429 response (no framework required).
def reject_response(retry_after_seconds, limit, remaining):
    return {
        "status": 429,
        "headers": {
            "Retry-After": str(retry_after_seconds),
            "X-RateLimit-Limit": str(limit),
            "X-RateLimit-Remaining": str(remaining),
        },
        "body": {"error": "rate_limited", "message": "slow down, please"},
    }

print(reject_response(retry_after_seconds=2, limit=60, remaining=0))

## 🗺️ Where we're going

| Notebook | Topic | Good for |
|---|---|---|
| 1 | Token bucket | API quotas that allow short bursts (AWS, Stripe) |
| 2 | Leaky bucket — shaper vs limiter | Smoothing traffic to a fragile downstream |
| 3 | Fixed vs sliding window | Fair, boundary-safe per-window limits |
| 4 | Distributed limits + client backoff | Multi-server deployments & good citizens |
| 5 | Concurrency limits & production practice | In-flight caps, cost-based limits, keys, fail-open |

Each notebook shows a **bad first try**, then a **better version**, and measures the difference rather than asserting it.